In [ ]:
# 1. List all files in the directory
files = dbutils.fs.ls("/Volumes/main/lufthansa/landing_zone/")

# 2. Filter for only the flight files you want
# This is basically "If the name starts with 'get_flight' and ends with '.json', keep it"
path_list = [f.path for f in files if "get_flight_on_routeLHR-STR" in f.name and f.name.endswith(".json")]

# 3. Pass the WHOLE LIST to Spark
# Spark is smart enough to accept a List of strings instead of just one string!
df_raw = spark.read.option("multiLine", "true").json(path_list)

print(f"Ingested {len(path_list)} files.")

: 

In [ ]:
from pyspark.sql.functions import col, explode

# 1. 'Explode' the array. 
# In your schema, 'Flight' is an ARRAY. Explode turns each item in that array into its own row.
# df_flights is a DataFrame object, not a list of strings. It is a Structured Collection. Every column has a specific type assigned by Spake when it "Inferred the Schema"
df_flights = df_raw.select(explode(col("FlightStatusResource.Flights.Flight")).alias("flight_data"))

# 2. Select the specific nested fields
df_silver = df_flights.select(
    col("flight_data.OperatingCarrier.AirlineID").alias("airline_code"),
    col("flight_data.OperatingCarrier.FlightNumber").alias("flight_number"),
    col("flight_data.Departure.AirportCode").alias("dep_airport_code"),
    col("flight_data.Arrival.AirportCode").alias("arr_airport_code"),
    col("flight_data.Departure.ScheduledTimeUTC.DateTime").alias("scheduled_dep_utc"),
    col("flight_data.Arrival.ScheduledTimeUTC.DateTime").alias("scheduled_arr_utc"),
    col("flight_data.FlightStatus.Code").alias("status"),
    col("flight_data.Equipment.AircraftCode").alias("aircraft_code"),
    col("flight_data.Departure.Terminal.Name").alias("dep_terminal")
)

#Fun fact, Spark doesn't actually do anything until you call an action. Like display(). This is called lazy evaluation.
display(df_silver)

: 

In [ ]:
# Save the cleaned data into my Catalog
# 'overwrite' ensures that if I run this again, it refreshes the table
df_silver.write.mode("overwrite").saveAsTable("main.lufthansa.silver_flight_status")

: 

In [ ]:
from pyspark.sql.functions import to_date

df_silver.withColumn("date", to_date("scheduled_dep_utc")).groupBy("date").count().orderBy("date").show()